# 第5章 LangGraph + LangSmith

LangChain の `AgentExecutor` の発展形である **LangGraph** と、実行トレースを可視化する **LangSmith** を体験します。

**所要時間**: 90〜120分

## 学ぶこと
1. prebuilt の `create_react_agent` で 1 行エージェントを作る
2. 手書きの `StateGraph` でノード/エッジを定義する
3. LangSmith にトレースを送り、エージェント内部の動きをダッシュボードで確認する
4. `MemorySaver` で会話履歴を保持し、複数ターンの対話を成立させる

## なぜ Agent から LangGraph に進むのか

第3章で見た `AgentExecutor` は便利ですが、内部はブラックボックスで以下のような要件には窮屈です:

- **分岐**: 「ツール結果がXならYに進む、ZならWに進む」
- **状態管理**: 「会話履歴」「中間結果」を明示的に持ちたい
- **人間レビュー**: 途中でユーザーの確認を挟みたい
- **並列実行**: 複数の処理を同時に走らせたい
- **再開可能性**: 途中で止めて、後から再開したい

LangGraph は **「ノード(処理)」と「エッジ(遷移)」のグラフ** としてエージェントを記述するライブラリです。
状態(State)を明示的に持つので、上記のような複雑な制御が書きやすくなります。

```
AgentExecutor の世界          LangGraph の世界
  ┌────────────┐              ┌──────────┐
  │ LLM ⇄ Tool │      →       │ State    │
  │ ループ      │              │ Node1 → Node2 → Node3
  └────────────┘              │ 分岐・並列・記憶可
                              └──────────┘
```

## 0. 準備 - 環境変数の読み込みと LangSmith 設定確認

In [2]:
import os
from dotenv import load_dotenv

# .env を読み込む。LANGSMITH_TRACING=true / LANGSMITH_API_KEY も読まれる
# 重要: LangSmith は環境変数を見て「自動的に」トレースを送る。明示的なコードは不要
load_dotenv()

AWS_REGION = os.environ["AWS_REGION"]
CHAT_MODEL_ID = os.environ["BEDROCK_CHAT_MODEL_ID"]

# LangSmith 設定の状況を確認
print("region:", AWS_REGION)
print("chat model:", CHAT_MODEL_ID)
print("LANGSMITH_TRACING:", os.environ.get("LANGSMITH_TRACING", "(unset)"))
print("LANGSMITH_PROJECT:", os.environ.get("LANGSMITH_PROJECT", "(unset)"))
print("LANGSMITH_ENDPOINT:", os.environ.get("LANGSMITH_ENDPOINT", "(unset)"))
print("LANGSMITH_API_KEY:",
      "set (length=%d)" % len(os.environ["LANGSMITH_API_KEY"])
      if os.environ.get("LANGSMITH_API_KEY") else "(unset - トレースは送信されません)")

region: ap-northeast-1
chat model: apac.amazon.nova-lite-v1:0
LANGSMITH_TRACING: true
LANGSMITH_PROJECT: work-project
LANGSMITH_ENDPOINT: https://apac.api.smith.langchain.com
LANGSMITH_API_KEY: set (length=51)


**API キーがまだ未設定なら**: 別途 [LangSmithセットアップガイド](../docs/05_langsmith_setup.md) を実施してください。
未設定のままでも Notebook は動きますが、トレース可視化のステップはスキップになります。

## 1. LLM とツールの準備

第3章と同じ2つのツールを用意します。LangGraph では `@tool` でツール化したものをそのまま使えます。

In [3]:
from datetime import datetime
from zoneinfo import ZoneInfo
from langchain_aws import ChatBedrockConverse
from langchain_core.tools import tool

# LLM 準備(これまでと同じ)
llm = ChatBedrockConverse(
    model=CHAT_MODEL_ID,
    region_name=AWS_REGION,
    temperature=0,
)

# ツール定義
@tool
def get_current_time(timezone: str = "Asia/Tokyo") -> str:
    """指定したタイムゾーンの現在時刻を ISO 8601 文字列で返す。"""
    return datetime.now(ZoneInfo(timezone)).isoformat(timespec="seconds")

@tool
def add_numbers(a: float, b: float) -> float:
    """2つの数値を加算した結果を返す。"""
    return a + b

tools = [get_current_time, add_numbers]

## 2. prebuilt の `create_react_agent` で 1 行エージェント

LangGraph には **「典型的な ReAct パターン(推論↔行動のループ)」をすぐ作れる prebuilt 関数** が用意されています。
第3章の `create_tool_calling_agent` + `AgentExecutor` の LangGraph 版と考えてください。

**戻り値**: コンパイル済みグラフ。`.invoke()` / `.stream()` できる Runnable。
**入力形式**: `{"messages": [{"role": "user", "content": "..."}]}` というメッセージのリストを渡す。

In [4]:
from langgraph.prebuilt import create_react_agent

# たった1行でエージェントが完成。第3章で書いた prompt / AgentExecutor 相当の構成が内部で組まれる
agent = create_react_agent(llm, tools)

# .invoke() の入力は messages リスト形式。OpenAI chat completion と互換
result = agent.invoke({
    "messages": [{"role": "user", "content": "いま東京は何時? それと 12.5 と 7.5 を足して。"}]
})

# 結果は messages リスト。最後が最終回答(AIMessage)
print("=== 全メッセージ履歴 ===")
for m in result["messages"]:
    m.pretty_print()

/home/vscode/.local/lib/python3.11/site-packages/langgraph/checkpoint/base/__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


=== 全メッセージ履歴 ===
================================ Human Message =================================

いま東京は何時? それと 12.5 と 7.5 を足して。
================================== Ai Message ==================================

[{'type': 'text', 'text': "<thinking>User wants to know the current time in Tokyo and the sum of 12.5 and 7.5. I can use the 'get_current_time' tool to find the current time in Tokyo and the 'add_numbers' tool to add the two numbers.</thinking>\n"}, {'type': 'tool_use', 'name': 'get_current_time', 'input': {'timezone': 'Asia/Tokyo'}, 'id': 'tooluse_D0GK5I3oHu3c8oo0yw5SjU'}, {'type': 'tool_use', 'name': 'add_numbers', 'input': {'a': 12.5, 'b': 7.5}, 'id': 'tooluse_qlorAM85INZW84vq1eIP0s'}]
Tool Calls:
  get_current_time (tooluse_D0GK5I3oHu3c8oo0yw5SjU)
 Call ID: tooluse_D0GK5I3oHu3c8oo0yw5SjU
  Args:
    timezone: Asia/Tokyo
  add_numbers (tooluse_qlorAM85INZW84vq1eIP0s)
 Call ID: tooluse_qlorAM85INZW84vq1eIP0s
  Args:
    a: 12.5
    b: 7.5
================================= Tool

## 3. LangSmith ダッシュボードで実行を確認

`.env` に `LANGSMITH_API_KEY` を設定していれば、上のセル実行が **自動的に LangSmith に送信されている** はずです。

**確認手順:**
1. https://smith.langchain.com/ にログイン
2. 左メニュー「Tracing Projects」 → `langchain-bedrock-handson`(`.env` の `LANGSMITH_PROJECT` 名)
3. 直近のトレースをクリック
4. ツリーを展開すると、以下のような階層が見えるはず:
   ```
   create_react_agent
   ├─ ChatBedrockConverse (1回目)         ← LLMが「ツールを呼ぶ」と判断
   ├─ get_current_time                   ← ツール実行
   ├─ ChatBedrockConverse (2回目)         ← 結果を見て「次もツール」と判断
   ├─ add_numbers                        ← ツール実行
   └─ ChatBedrockConverse (3回目)         ← 結果をまとめて最終回答
   ```
5. 各ノードをクリックすると **送信したプロンプト・受信した出力・トークン消費・所要時間** が見られる

verbose ログより圧倒的にデバッグしやすいですね。これが LangSmith の威力です。

## 4. 手書きの StateGraph で同じものを組む

prebuilt は便利ですが、自分でカスタマイズしたくなった時のために **StateGraph の基本** を理解しておきましょう。

### 基本要素

| 要素 | 役割 |
|---|---|
| **State** | グラフ全体で共有する状態(辞書のような型)|
| **Node** | 状態を受け取り、状態の更新を返す関数 |
| **Edge** | ノード間の遷移。条件付き分岐も書ける |
| **`START` / `END`** | グラフの入り口/出口の特殊ノード |

### 今回作るグラフ

```
      START
        ↓
    call_model ←──┐
        ↓         │
  tools_condition │   (ツール呼び出しがあれば tools へ)
      ↓     ↓     │
    END    tools ─┘   (なければ END)
```

ReAct パターンの最小構成です。LLMノード(`call_model`)とツール実行ノード(`tools`)を、条件付きエッジで結ぶだけ。

In [5]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

# State: 今回は MessagesState(組み込み)を使う
# これは {"messages": [...]} という 1 キーの辞書型で、追加時には自動マージされる

# LLM にツールを bind しておく(これにより LLM 応答に tool_calls が含まれるようになる)
llm_with_tools = llm.bind_tools(tools)

# ノード1: LLM を呼ぶ
# 引数 state は State 型、戻り値は「更新分の dict」(messages キーに新しい応答を追加)
def call_model(state: MessagesState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}  # MessagesState は自動で append される

# グラフを組み立てる
builder = StateGraph(MessagesState)

# ノード追加: 1つ目は自分で書いた関数、2つ目は prebuilt の ToolNode(複数ツールを実行できる)
builder.add_node("call_model", call_model)
builder.add_node("tools", ToolNode(tools))

# エッジ追加
builder.add_edge(START, "call_model")

# 条件付きエッジ: tools_condition は「LLMの最新応答に tool_calls があるか」を見て、
#                ある → "tools" へ、ない → END へルーティングしてくれる prebuilt
builder.add_conditional_edges("call_model", tools_condition)

# ツール実行後は LLM に戻ってループ
builder.add_edge("tools", "call_model")

# .compile() で実行可能なグラフ(Runnable)になる
graph = builder.compile()

# 実行
result = graph.invoke({
    "messages": [{"role": "user", "content": "今 UTC で何時? その後 3.14 と 2.86 を足して。"}]
})

print("=== 全メッセージ履歴 ===")
for m in result["messages"]:
    m.pretty_print()

=== 全メッセージ履歴 ===
================================ Human Message =================================

今 UTC で何時? その後 3.14 と 2.86 を足して。
================================== Ai Message ==================================

[{'type': 'text', 'text': '<thinking>まず、UTC で現在の時刻を取得します。その後、3.14 と 2.86 を加算します。</thinking>\n'}, {'type': 'tool_use', 'name': 'get_current_time', 'input': {'timezone': 'UTC'}, 'id': 'tooluse_8V0bwqLUWcunj0En53L0xd'}, {'type': 'tool_use', 'name': 'add_numbers', 'input': {'a': 3.14, 'b': 2.86}, 'id': 'tooluse_az7XBwCzi0CFnWfF2uKVPO'}]
Tool Calls:
  get_current_time (tooluse_8V0bwqLUWcunj0En53L0xd)
 Call ID: tooluse_8V0bwqLUWcunj0En53L0xd
  Args:
    timezone: UTC
  add_numbers (tooluse_az7XBwCzi0CFnWfF2uKVPO)
 Call ID: tooluse_az7XBwCzi0CFnWfF2uKVPO
  Args:
    a: 3.14
    b: 2.86
================================= Tool Message =================================
Name: get_current_time

2026-05-27T15:26:34+00:00
================================= Tool Message ======================

## 5. グラフを可視化する

組み上げたグラフの構造をテキスト(ASCII)で確認できます。

Mermaid 形式の出力も可能で、Markdown ビューアやドキュメントに貼り付けて使えます。

In [6]:
# Mermaid 形式の文字列を出力(GitHub の Markdown 等にそのまま貼れる)
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	call_model(call_model)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> call_model;
	call_model -.-> __end__;
	call_model -.-> tools;
	tools --> call_model;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [7]:
# ASCII 形式での可視化(grandalf パッケージが入っていれば動作)
try:
    print(graph.get_graph().draw_ascii())
except Exception as e:
    print("ASCII 描画には grandalf が必要です。Mermaid 出力(上のセル)を使ってください。")
    print(f"  詳細: {e}")

        +-----------+         
        | __start__ |         
        +-----------+         
               *              
               *              
               *              
        +------------+        
        | call_model |        
        +------------+        
          .         *         
        ..           **       
       .               *      
+---------+         +-------+ 
| __end__ |         | tools | 
+---------+         +-------+ 


## 6. MemorySaver で会話履歴を保持する

ここまでは「1回の質問→回答」で完結していました。実用的なチャットアプリでは **過去の会話を覚えていてほしい** ですよね。

LangGraph は **Checkpointer** という仕組みで、各実行のステートをスナップショット保存できます。
`MemorySaver` はその最も簡単な実装(プロセスメモリに保存)。

**使い方**:
1. グラフを `compile(checkpointer=memory)` で再コンパイル
2. 実行時に `config={"configurable": {"thread_id": "任意のID"}}` を渡す
3. 同じ `thread_id` で次の invoke を呼ぶと、前回の messages が自動で復元される

本番では `SqliteSaver` / `PostgresSaver` 等の永続化版に差し替えるだけで OK。

In [8]:
from langgraph.checkpoint.memory import MemorySaver

# プロセスメモリ上に履歴を保存するシンプルなチェックポインタ
memory = MemorySaver()

# 同じ builder からチェックポインタ付きでコンパイル
graph_with_memory = builder.compile(checkpointer=memory)

# thread_id を指定して 1 ターン目
config = {"configurable": {"thread_id": "demo-conversation-001"}}

result1 = graph_with_memory.invoke(
    {"messages": [{"role": "user", "content": "はじめまして。私の名前は田中です。"}]},
    config,
)
print("--- 1ターン目 ---")
result1["messages"][-1].pretty_print()

--- 1ターン目 ---
================================== Ai Message ==================================

<thinking>User has introduced themselves and provided their name. There is no specific tool required for this interaction, so I will respond directly to the User.</thinking>
はじめまして、田中さん。私はAIアシスタントです。何かお手伝いできることがあれば、いつでも私に知らせてください。


In [ ]:
# 同じ thread_id で 2 ターン目。前のやり取り(自己紹介)が引き継がれているはず
# 「覚えていますか?」と聞くと、Nova Lite は"AIに記憶はない"と前置きしがち。
# 「私の名前で挨拶して」のように 履歴を使うことを促す質問の方が、効果がはっきり見える
result2 = graph_with_memory.invoke(
    {"messages": [{"role": "user", "content": "私の名前で挨拶してください。"}]},
    config,  # 同じ thread_id
)
print("--- 2ターン目 ---")
result2["messages"][-1].pretty_print()

--- 2ターン目 ---
================================== Ai Message ==================================

<thinking>User has asked to be greeted by their name. I will respond directly to the User.</thinking>
田中さん、こんにちは！お会いできて嬉しいです。何かお手伝いできることがあれば、いつでも私に知らせてください。


In [10]:
# 異なる thread_id にすると、独立した会話になる
# 上と同じ質問でも、こちらは履歴が無いので「お名前を教えてください」のような応答になるはず
result3 = graph_with_memory.invoke(
    {"messages": [{"role": "user", "content": "私の名前で挨拶してください。"}]},
    {"configurable": {"thread_id": "another-conversation"}},  # 別の thread_id
)
print("--- 別 thread ---")
result3["messages"][-1].pretty_print()

--- 別 thread ---
================================== Ai Message ==================================

<thinking> The User is asking if I remember their name. I do not have the capability to remember personal information from previous interactions. I should inform the User that I cannot remember their name from previous interactions. </thinking>
I'm sorry, but I do not have the capability to remember personal information from previous interactions. How can I assist you today?


## まとめ

- **LangGraph** は AgentExecutor の発展形。`State / Node / Edge` でエージェントの内部構造を明示的に書ける
- `create_react_agent` で素早く動かす → 必要に応じて手書き StateGraph に展開、というステップが王道
- `MessagesState` + `ToolNode` + `tools_condition` の組み合わせが最小 ReAct パターン
- **LangSmith** は環境変数を設定するだけでトレースが自動送信される。デバッグが激変する
- **MemorySaver** + `thread_id` で会話履歴を持てる。本番は永続化版に差し替え

次は [06 RAG深掘り](../06_rag_advanced/) で、第2章で作った素朴な RAG を LangGraph 化しつつ、検索精度を上げていきます。